<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Dataset-2-Preprocessing---automating-file-upload-%26-adding-error-checking-for-NaN-values-if-any/Group_12_Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Initial Test Commit

## STEP 1: Importing Packages

In [1]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [2]:
#MUST BE RUN inorder for file download to work
!pip install gdown

#Download dataset csv into runtime from google drive link
import gdown

url = "https://drive.google.com/uc?id=1AmoAC0dGGWuzhL8yWk0EjjpTGJ-FvqGP"
gdown.download(url, "card_transdata.csv", quiet=False)

#loading csv file into pandas dataframe
df = pd.read_csv('card_transdata.csv')

Downloading...
From: https://drive.google.com/uc?id=1AmoAC0dGGWuzhL8yWk0EjjpTGJ-FvqGP
To: /content/card_transdata.csv
100%|██████████| 76.3M/76.3M [00:01<00:00, 75.7MB/s]


In [4]:
#printing dataset information (how many rows, columns etc)
print(df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(df.isnull().sum())
print('----------------------------------------------------------')

# Show ONLY columns with missing values
print("Columns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print('----------------------------------------------------------')

#Dropping rows with missing values
df = df.dropna()

#Verifying that those rows have been dropped and there are no missing values
print("Total missing values after dropping:")
print(df.isnull().sum().sum())  # should be 0
print('----------------------------------------------------------')

#printing first 5 rows of data to ensure it correlates with datafile
print(df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(df['fraud'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB
None
----------------------------------------------------------
distance_from_home                0
distance_from_last_transaction    0
ratio_to_median_purchase_price    0
repeat_retailer                   0
used_chip 

## Step 3: Split Data into Features & Target

In [ ]:
#Splitting features and target
X = df.drop('fraud', axis=1)
y = df['fraud']

## Step 4: Decision Tree Model using K Fold Cross Validation


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Decision Tree Model
model_dt = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Store results
all_results = []

# Manual KFold Loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):

    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Train model
    model_dt.fit(X_train_resampled, y_train_resampled)

    # Predict on test data
    y_pred = model_dt.predict(X_test)
    y_prob = model_dt.predict_proba(X_test)[:, 1]

    #Performance metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df = pd.DataFrame({
        "Metric": list(results.keys()),
        "Value": [round(value, 4) for value in results.values()]
    })

    print(results_df)

    print(f"\n============================================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print(f"\n============================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Appending results
    all_results.append(results)

# Taking average of all results
results_df = pd.DataFrame(all_results)

avg_results = results_df.mean()
std_results = results_df.std()

final_df = pd.DataFrame({
    "Metric": avg_results.index,
    "Average Value": [round(val, 4) for val in avg_results.values],
    "Std Dev": [round(val, 4) for val in std_results.values]
})

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df)


================== Fold 1 ==================
      Metric   Value
0   Accuracy  0.9991
1  Precision  0.9932
2     Recall  0.9970
3   F1 Score  0.9951
4    ROC-AUC  1.0000

Confusion Matrix:
[[91200    60]
 [   26  8714]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9997    0.9993    0.9995     91260
         1.0     0.9932    0.9970    0.9951      8740

    accuracy                         0.9991    100000
   macro avg     0.9964    0.9982    0.9973    100000
weighted avg     0.9991    0.9991    0.9991    100000


================== Fold 2 ==================
      Metric   Value
0   Accuracy  0.9989
1  Precision  0.9917
2     Recall  0.9960
3   F1 Score  0.9938
4    ROC-AUC  0.9999

Confusion Matrix:
[[91187    73]
 [   35  8705]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9996    0.9992    0.9994     91260
         1.0     0.9917    0.9960    0.9938      8740

    accuracy     

## Step 6: Deep Neural Network

In [ ]:
#Runtime approx. 35mins

#Importing relavent packages
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

#Creating DNN Model
def create_dnn_model(input_dim):
  model = models.Sequential([
    Input(shape = (input_dim,)),

    layers.Dense(32, activation = 'relu'),
    layers.BatchNormalization(),                                                    #Normalize outputs

    layers.Dense(16, activation = 'relu'),
    layers.Dropout(0.2),                                                           #Prevents overfitting (20% dropoff)

    layers.Dense(8, activation = 'relu'),

    layers.Dense(1, activation = 'sigmoid')
  ])

  model.compile(
      optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
      loss = 'binary_crossentropy',
      metrics = [
          tf.keras.metrics.Precision(),
          tf.keras.metrics.Recall(),
          tf.keras.metrics.AUC()
      ]
  )
  return model


##K-Fold Setup, Training Model, Predicting and Metrics

#K-Fold setup
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
all_results = []


#Manual K-Fold loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):
    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #Scaling Data
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)


    #Training Model
    model = create_dnn_model(input_dim = X_train_resampled.shape[1])

    history = model.fit(
        X_train_resampled, y_train_resampled,
        epochs = 10,
        batch_size = 256,
        validation_split = 0.2,
        verbose = 1
    )

    #Predicting
    y_prob = model.predict(X_test_scaled, verbose = 1).flatten()
    y_pred = (y_prob > 0.5).astype(float)

    #Performance Metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df_dnn_dataset1 = pd.DataFrame({"Metric": list(results.keys()), "Value": [round(value, 4) for value in results.values()]})
    print(results_df_dnn_dataset1)

    print("\n Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\n Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))


    #Appending all results
    all_results.append(results)

#Tabulated Average of Results
results_df_dnn_dataset1  = pd.DataFrame(all_results)
avg_results_dnn_dataset1 = results_df_dnn_dataset1.mean()
std_results_dnn_dataset1 = results_df_dnn_dataset1.std()
final_df_dnn_dataset1    = pd.DataFrame({"Metric": avg_results_dnn_dataset1.index,
                                         "Average Value": [round(val, 4) for val in avg_results_dnn_dataset1.values],
                                         "Std Dev": [round(val, 4) for val in std_results_dnn_dataset1.values],
                                         })

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df_dnn_dataset1.to_string(index=False))



================== Fold 1 ==================
Epoch 1/10
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - auc: 0.9986 - loss: 0.0434 - precision: 0.9759 - recall: 0.9827 - val_auc: 0.0000e+00 - val_loss: 0.0130 - val_precision: 1.0000 - val_recall: 0.9988
Epoch 2/10
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - auc: 0.9997 - loss: 0.0179 - precision: 0.9882 - recall: 0.9942 - val_auc: 0.0000e+00 - val_loss: 0.0041 - val_precision: 1.0000 - val_recall: 0.9995
Epoch 3/10
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - auc: 0.9997 - loss: 0.0153 - precision: 0.9898 - recall: 0.9951 - val_auc: 0.0000e+00 - val_loss: 0.0127 - val_precision: 1.0000 - val_recall: 0.9983
Epoch 4/10
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - auc: 0.9998 - loss: 0.0134 - precision: 0.9910 - recall: 0.9957 - val_auc: 0.0000e+00 - val_loss: 0.0031 - val_precision: 1.0000 - val_recall: 0.9998
Epoch 5/10
5134/5134 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - auc: 0.9998 - loss: 0.0125 - precision: 0.9917 - recall: 0.9960 - val

## Step 7: Logistic Regression

###Step 7a: Train the Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

LogisticRegression(max_iter=1000, random_state=42)

###Step 7b: Making Predictions

In [ ]:
#Predictions
y_pred = lr_model.predict(X_test)
y_prob = lr_model.predict_proba(X_test)[:, 1]

###Step 7c: Evaluating the Model Using Performance Metrics

In [ ]:
#Evaluating the model using performance metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Calculating performance metrics
results = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob)
}

#Printing results in an easy to read table
results_df = pd.DataFrame({"Metric": list(results.keys()), "Value": [round(value, 4) for value in results.values()]})
print(results_df)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

      Metric   Value
0   Accuracy  0.9344
1  Precision  0.5757
2     Recall  0.9484
3   F1 Score  0.7164
4    ROC-AUC  0.9794
Confusion Matrix:
[[85148  6111]
 [  451  8290]]
Classification Report:
              precision    recall  f1-score   support

         0.0     0.9947    0.9330    0.9629     91259
         1.0     0.5757    0.9484    0.7164      8741

    accuracy                         0.9344    100000
   macro avg     0.7852    0.9407    0.8397    100000
weighted avg     0.9581    0.9344    0.9414    100000

